In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

import logging
import warnings
import os

# Reduz logs gerais
logging.getLogger().setLevel(logging.CRITICAL)
for nome in ["matplotlib", "PIL", "numexpr", "tensorflow", "absl"]:
    logging.getLogger(nome).setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Base de dados: E-commerce (Comportamento de Compra)
dados = pd.DataFrame({
    "Data_Competencia": ["2026-04", "2026-04", "2026-04", "2026-05", "2026-05", "2026-05", "2026-04", "2026-05", "2026-04", "2026-05"],
    "Cliente": ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10"],
    "Nivel_Renda": ["Alta", "Alta", "Média", "Baixa", "Baixa", "Alta", "Média", "Baixa", "Alta", "Média"],
    "Perfil_Compra": ["Eletronicos", "Vestuario", "Vestuario", "Alimentos", "Alimentos", "Eletronicos", "Alimentos", "Vestuario", "Eletronicos", "Vestuario"],
    "Cupom_Desconto": ["Sim", "Sim", "Nao", "Nao", "Nao", "Sim", "Nao", "Nao", "Sim", "Sim"],
    "Conversao_Venda": ["Comprou", "Comprou", "Nao_Comprou", "Nao_Comprou", "Nao_Comprou", "Comprou", "Nao_Comprou", "Nao_Comprou", "Comprou", "Comprou"]
})

display(dados)

,Data_Competencia,Cliente,Nivel_Renda,Perfil_Compra,Cupom_Desconto,Conversao_Venda
0,2026-04,C1,Alta,Eletronicos,Sim,Comprou
1,2026-04,C2,Alta,Vestuario,Sim,Comprou
2,2026-04,C3,Média,Vestuario,Nao,Nao_Comprou
3,2026-05,C4,Baixa,Alimentos,Nao,Nao_Comprou
4,2026-05,C5,Baixa,Alimentos,Nao,Nao_Comprou
5,2026-05,C6,Alta,Eletronicos,Sim,Comprou
6,2026-04,C7,Média,Alimentos,Nao,Nao_Comprou
7,2026-05,C8,Baixa,Vestuario,Nao,Nao_Comprou
8,2026-04,C9,Alta,Eletronicos,Sim,Comprou
9,2026-05,C10,Média,Vestuario,Sim,Comprou


In [2]:
transacoes = []

for _, linha in dados.iterrows():
    transacao = [
        f"Data={linha['Data_Competencia']}",
        f"Renda={linha['Nivel_Renda']}",
        f"Perfil={linha['Perfil_Compra']}",
        f"Cupom={linha['Cupom_Desconto']}",
        f"Conversao={linha['Conversao_Venda']}"
    ]
    transacoes.append(transacao)

te = TransactionEncoder()
matriz = te.fit(transacoes).transform(transacoes)
dados_transformados = pd.DataFrame(matriz, columns=te.columns_)

display(dados_transformados.head())

,Conversao=Comprou,Conversao=Nao_Comprou,Cupom=Nao,Cupom=Sim,Data=2026-04,Data=2026-05,Perfil=Alimentos,Perfil=Eletronicos,Perfil=Vestuario,Renda=Alta,Renda=Baixa,Renda=Média
0,True,False,False,True,True,False,False,True,False,True,False,False
1,True,False,False,True,True,False,False,False,True,True,False,False
2,False,True,True,False,True,False,False,False,True,False,False,True
3,False,True,True,False,False,True,True,False,False,False,True,False
4,False,True,True,False,False,True,True,False,False,False,True,False


In [3]:
itemsets_frequentes = apriori(
    dados_transformados,
    min_support=0.3,
    use_colnames=True
)

display(itemsets_frequentes.sort_values(by="support", ascending=False))

,support,itemsets
0,0.5,(Conversao=Comprou)
1,0.5,(Conversao=Nao_Comprou)
2,0.5,(Cupom=Nao)
3,0.5,(Cupom=Sim)
4,0.5,(Data=2026-04)
5,0.5,(Data=2026-05)
16,0.5,"(Conversao=Nao_Comprou, Cupom=Nao)"
12,0.5,"(Conversao=Comprou, Cupom=Sim)"
25,0.4,"(Renda=Alta, Cupom=Sim)"
15,0.4,"(Conversao=Comprou, Renda=Alta)"


In [4]:
regras = association_rules(
    itemsets_frequentes,
    metric="confidence",
    min_threshold=0.7
)

regras_resumidas = regras[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].copy()

def formatar_conjunto(conjunto):
    return ", ".join(list(conjunto))

regras_resumidas["Regra"] = regras_resumidas.apply(
    lambda linha: f"{formatar_conjunto(linha['antecedents'])} → {formatar_conjunto(linha['consequents'])}",
    axis=1
)

display(regras_resumidas[["Regra", "support", "confidence", "lift"]].sort_values(
    by="confidence",
    ascending=False
))

,Regra,support,confidence,lift
0,Conversao=Comprou → Cupom=Sim,0.5,1.00,2.0
1,Cupom=Sim → Conversao=Comprou,0.5,1.00,2.0
2,Perfil=Eletronicos → Conversao=Comprou,0.3,1.00,2.0
4,Renda=Alta → Conversao=Comprou,0.4,1.00,2.0
5,Conversao=Nao_Comprou → Cupom=Nao,0.5,1.00,2.0
...,...,...,...,...
69,"Renda=Alta, Cupom=Sim → Conversao=Comprou, Dat...",0.3,0.75,2.5
62,"Renda=Alta → Perfil=Eletronicos, Cupom=Sim",0.3,0.75,2.5
63,"Conversao=Comprou, Renda=Alta, Cupom=Sim → Dat...",0.3,0.75,1.5
82,"Renda=Alta → Conversao=Comprou, Cupom=Sim, Per...",0.3,0.75,2.5


In [5]:
# Regras onde o cliente efetivou a compra
regras_comprou = regras_resumidas[
    regras_resumidas["consequents"].apply(lambda x: "Conversao=Comprou" in x)
]

print("=== Padrões que geram Vendas ===")
display(regras_comprou[["Regra", "support", "confidence", "lift"]])

# Regras onde o cliente não comprou
regras_nao_comprou = regras_resumidas[
    regras_resumidas["consequents"].apply(lambda x: "Conversao=Nao_Comprou" in x)
]

print("\n=== Padrões de Abandono/Não Compra ===")
display(regras_nao_comprou[["Regra", "support", "confidence", "lift"]])

=== Padrões que geram Vendas ===


,Regra,support,confidence,lift
1,Cupom=Sim → Conversao=Comprou,0.5,1.00,2.0
2,Perfil=Eletronicos → Conversao=Comprou,0.3,1.00,2.0
4,Renda=Alta → Conversao=Comprou,0.4,1.00,2.0
19,"Cupom=Sim, Data=2026-04 → Conversao=Comprou",0.3,1.00,2.0
21,"Perfil=Eletronicos, Cupom=Sim → Conversao=Comprou",0.3,1.00,2.0
22,"Perfil=Eletronicos → Conversao=Comprou, Cupom=Sim",0.3,1.00,2.0
25,"Renda=Alta, Cupom=Sim → Conversao=Comprou",0.4,1.00,2.0
27,"Renda=Alta → Conversao=Comprou, Cupom=Sim",0.4,1.00,2.0
28,"Cupom=Sim → Conversao=Comprou, Renda=Alta",0.4,0.80,2.0
31,"Renda=Alta, Data=2026-04 → Conversao=Comprou",0.3,1.00,2.0



=== Padrões de Abandono/Não Compra ===


,Regra,support,confidence,lift
6,Cupom=Nao → Conversao=Nao_Comprou,0.5,1.0,2.000000
7,Perfil=Alimentos → Conversao=Nao_Comprou,0.3,1.0,2.000000
8,Renda=Baixa → Conversao=Nao_Comprou,0.3,1.0,2.000000
39,"Data=2026-05, Cupom=Nao → Conversao=Nao_Comprou",0.3,1.0,2.000000
41,"Perfil=Alimentos, Cupom=Nao → Conversao=Nao_Co...",0.3,1.0,2.000000
42,"Perfil=Alimentos → Conversao=Nao_Comprou, Cupo...",0.3,1.0,2.000000
44,"Renda=Baixa, Cupom=Nao → Conversao=Nao_Comprou",0.3,1.0,2.000000
45,"Renda=Baixa → Conversao=Nao_Comprou, Cupom=Nao",0.3,1.0,2.000000
48,"Renda=Baixa, Data=2026-05 → Conversao=Nao_Comprou",0.3,1.0,2.000000
49,"Renda=Baixa → Conversao=Nao_Comprou, Data=2026-05",0.3,1.0,3.333333
